# Machine Learning Modeling
## Ethiopian Household Wealth Prediction

**Objective:** Train and compare ML models to predict household consumption expenditure.

**Models trained (with justification):**

- **Ridge Regression**: Baseline linear model with L2 regularization. Handles multicollinearity.
- **Lasso Regression**: L1 regularization performs automatic feature selection.
- **Random Forest**: Ensemble of trees reduces variance; handles non-linear relationships.
- **Gradient Boosting**: Sequential ensemble; often best performer on tabular data.
- **XGBoost**: Industry standard; built-in regularization and missing-value handling.
- **LightGBM**: Faster than XGBoost for larger datasets.

**Evaluation Metrics:** R², RMSE, MAE, and cross-validated R².

**Special Analyses:** Regional performance, COVID-19 impact, and feature importance.

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os, joblib, warnings
warnings.filterwarnings('ignore')
# Add src to path
sys.path.append(os.path.abspath('..'))
from src.modeling import WealthPredictor
from src.visualization import WealthVisualizer
# Settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 200)
%matplotlib inline
sns.set_palette("husl")
viz = WealthVisualizer()
print("Setup complete!")

## 2. Load Processed Data

In [ ]:
PROCESSED_PATH = '../data/processed/'
# Load train/val/test splits
X_train = pd.read_csv(f'{PROCESSED_PATH}X_train.csv')
X_val = pd.read_csv(f'{PROCESSED_PATH}X_val.csv')
X_test = pd.read_csv(f'{PROCESSED_PATH}X_test.csv')
y_train = pd.read_csv(f'{PROCESSED_PATH}y_train.csv').iloc[:, 0]
y_val = pd.read_csv(f'{PROCESSED_PATH}y_val.csv').iloc[:, 0]
y_test = pd.read_csv(f'{PROCESSED_PATH}y_test.csv').iloc[:, 0]
print("DATA SPLITS:")
print(f"  Training:   {X_train.shape[0]:,} samples")
print(f"  Validation: {X_val.shape[0]:,} samples")
print(f"  Test:       {X_test.shape[0]:,} samples")
print(f"  Features:   {X_train.shape[1]}")
print(f"\nTarget (Log Consumption):")
print(f"  Train: mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"  Test:  mean={y_test.mean():.3f}, std={y_test.std():.3f}")
# Quick sanity check
print(f"\nFeature columns (first 20):")
print(X_train.columns.tolist()[:20])

## 3. Train All Models

In [ ]:
# Initialize predictor with all models
predictor = WealthPredictor(random_state=42)
print("TRAINING ALL MODELS...")
print("=" * 60)
# Train and evaluate on test set
results = predictor.train_evaluate(X_train, y_train, X_test, y_test)
print("\n" + "=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)
display(results.style.background_gradient(subset=['R2', 'CV_R2'], cmap='RdYlGn'))

## 4. Visualize Model Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
results_sorted = results.sort_values('R2', ascending=True)
axes[0, 0].barh(results_sorted['Model'], results_sorted['R2'], color='steelblue')
axes[0, 0].set_xlabel('R² Score')
axes[0, 0].set_title('R² Score by Model (higher is better)')
for i, v in enumerate(results_sorted['R2']):
    axes[0, 0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
axes[0, 1].barh(results_sorted['Model'], results_sorted['RMSE'], color='coral')
axes[0, 1].set_xlabel('RMSE')
axes[0, 1].set_title('RMSE by Model (lower is better)')
for i, v in enumerate(results_sorted['RMSE']):
    axes[0, 1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
x = np.arange(len(results))
width = 0.35
axes[1, 0].bar(x - width/2, results['R2'], width, label='Test R²', color='steelblue')
axes[1, 0].bar(x + width/2, results['CV_R2'], width, label='CV R²', color='lightgreen')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(results['Model'], rotation=45, ha='right')
axes[1, 0].set_ylabel('R² Score')
axes[1, 0].set_title('Test R² vs Cross-Validation R²')
axes[1, 0].legend()
axes[1, 1].barh(results_sorted['Model'], results_sorted['MAE'], color='mediumpurple')
axes[1, 1].set_xlabel('MAE')
axes[1, 1].set_title('Mean Absolute Error by Model')
for i, v in enumerate(results_sorted['MAE']):
    axes[1, 1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Best Model Analysis

In [ ]:
best_model_name = predictor.best_model_name
best_model = predictor.best_model
print(f"BEST MODEL: {best_model_name}")
print(f"Test R²: {results[results['Model']==best_model_name]['R2'].values[0]:.4f}")
print(f"Test RMSE: {results[results['Model']==best_model_name]['RMSE'].values[0]:.4f}")
# Make predictions with best model
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

## 6. Actual vs Predicted Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(y_test, y_pred_log, alpha=0.3, s=5, color='steelblue')
lims = [min(y_test.min(), y_pred_log.min()), max(y_test.max(), y_pred_log.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_xlabel('Actual Log Consumption')
axes[0].set_ylabel('Predicted Log Consumption')
axes[0].set_title(f'Actual vs Predicted (Log Scale)\nR² = {results[results["Model"]==best_model_name]["R2"].values[0]:.4f}')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].scatter(y_true, y_pred, alpha=0.3, s=5, color='green')
lims_etb = [0, max(y_true.max(), y_pred.max())]
axes[1].plot(lims_etb, lims_etb, 'r--', linewidth=1.5, label='Perfect Prediction')
axes[1].set_xlabel('Actual Consumption (ETB)')
axes[1].set_ylabel('Predicted Consumption (ETB)')
axes[1].set_title('Actual vs Predicted (ETB)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
residuals = y_test - y_pred_log
axes[2].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='coral', density=True)
axes[2].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero Error')
from scipy import stats
x_range = np.linspace(residuals.min(), residuals.max(), 100)
axes[2].plot(x_range, stats.norm.pdf(x_range, residuals.mean(), residuals.std()), 'b-', linewidth=2, label='Normal Distribution')
axes[2].set_xlabel('Residual (Actual - Predicted)')
axes[2].set_ylabel('Density')
axes[2].set_title(f'Residual Distribution\nMean={residuals.mean():.4f}, Std={residuals.std():.4f}')
axes[2].legend()
plt.suptitle(f'Best Model: {best_model_name} - Prediction Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
importance = predictor.get_feature_importance(best_model_name, top_n=20)
if importance is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    top_features = importance.head(20)
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_features)))
    axes[0].barh(range(len(top_features)), top_features.values, color=colors)
    axes[0].set_yticks(range(len(top_features)))
    axes[0].set_yticklabels(top_features.index, fontsize=9)
    axes[0].set_xlabel('Importance')
    axes[0].set_title(f'Top 20 Features - {best_model_name}')
    axes[0].invert_yaxis()
    top10 = importance.head(10)
    other_importance = importance.iloc[10:].sum()
    pie_data = list(top10.values) + [other_importance]
    pie_labels = list(top10.index) + ['Other Features']
    axes[1].pie(pie_data, labels=pie_labels, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 7})
    axes[1].set_title('Feature Importance Distribution (Top 10 + Others)')
    plt.suptitle(f'What Drives Household Wealth in Ethiopia?', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print("\nTOP 20 FEATURES:")
    display(importance.to_frame('Importance'))
else:
    print("Feature importance not available for this model type.")

## 8. Hyperparameter Tuning

In [ ]:
print("HYPERPARAMETER TUNING")
print("=" * 60)
if best_model_name in ['XGBoost', 'LightGBM', 'Random Forest', 'Gradient Boosting']:
    print(f"\nTuning {best_model_name}...")
    print("This may take a few minutes...\n")
    tuned_model = predictor.hyperparameter_tune(X_train, y_train, model_name=best_model_name, cv=3)
    if tuned_model is not None:
        y_pred_tuned = tuned_model.predict(X_test)
        r2_tuned = 1 - np.sum((y_test - y_pred_tuned)**2) / np.sum((y_test - y_test.mean())**2)
        rmse_tuned = np.sqrt(np.mean((y_test - y_pred_tuned)**2))
        original_r2 = results[results['Model']==best_model_name]['R2'].values[0]
        improvement = r2_tuned - original_r2
        print(f"  Improvement: {improvement:+.4f} ({improvement/original_r2*100:+.1f}%)")
        if r2_tuned > original_r2:
            predictor.best_model = tuned_model
            predictor.models[best_model_name] = tuned_model
            print(f"  ✓ Tuned model is BETTER - Updated best model")
        else:
            print(f"  → Keeping original model (no improvement from tuning)")
else:
    print(f"  Skipping tuning for {best_model_name} (linear model)")

## 9. Regional Performance Analysis

In [ ]:
print("REGIONAL MODEL PERFORMANCE")
print("=" * 60)
df_full = pd.read_csv(f'{PROCESSED_PATH}processed_data.csv', low_memory=False)
region_col = None
for c in df_full.columns:
    if 'region' in c.lower() and df_full[c].nunique() > 1 and df_full[c].nunique() < 20:
        region_col = c
        break
if region_col:
    print(f"Region column: {region_col}")
    test_indices = X_test.index
    test_regions = df_full.loc[test_indices, region_col] if region_col in df_full.columns else None
    if test_regions is not None:
        test_results = pd.DataFrame({'region': test_regions.values, 'actual': y_test.values, 'predicted': y_pred_log})
        regional_r2 = {}
        for region in sorted(test_results['region'].dropna().unique()):
            region_data = test_results[test_results['region'] == region]
            if len(region_data) >= 30:
                ss_res = np.sum((region_data['actual'] - region_data['predicted'])**2)
                ss_tot = np.sum((region_data['actual'] - region_data['actual'].mean())**2)
                r2 = 1 - ss_res / ss_tot
                regional_r2[region] = {'R²': r2, 'Samples': len(region_data), 'Mean_Actual': np.expm1(region_data['actual']).mean(), 'Mean_Predicted': np.expm1(region_data['predicted']).mean()}
        if regional_r2:
            regional_df = pd.DataFrame(regional_r2).T.sort_values('R²', ascending=False)
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            axes[0].barh(regional_df.index, regional_df['R²'], color='steelblue')
            axes[0].set_xlabel('R² Score')
            axes[0].set_title('Model Performance by Region')
            axes[0].axvline(results[results['Model']==best_model_name]['R2'].values[0], color='red', linestyle='--', label='National Average')
            axes[0].legend()
            regional_df[['Mean_Actual', 'Mean_Predicted']].plot(kind='barh', ax=axes[1])
            axes[1].set_xlabel('Consumption (ETB)')
            axes[1].set_title('Actual vs Predicted Mean by Region')
            plt.suptitle(f'Regional Analysis - {best_model_name}', fontweight='bold')
            plt.tight_layout()
            plt.show()
            print("\nRegional R² Scores:")
            display(regional_df.round(3))
    else:
        print("Region information not available in the processed data.")
else:
    print("No region column found in data. Skipping regional analysis.")

## 10. COVID-19 Impact Analysis

In [ ]:
print("COVID-19 IMPACT ANALYSIS")
print("=" * 60)
if 'post_covid' in X_test.columns or 'wave' in X_test.columns:
    if 'post_covid' in X_test.columns:
        pre_covid_mask = X_test['post_covid'] == 0
        post_covid_mask = X_test['post_covid'] == 1
    else:
        pre_covid_mask = X_test['wave'] <= 4
        post_covid_mask = X_test['wave'] == 5
    pre_covid_actual = y_test[pre_covid_mask]
    pre_covid_pred = y_pred_log[pre_covid_mask]
    post_covid_actual = y_test[post_covid_mask]
    post_covid_pred = y_pred_log[post_covid_mask]
    pre_r2 = 1 - np.sum((pre_covid_actual - pre_covid_pred)**2) / np.sum((pre_covid_actual - pre_covid_actual.mean())**2)
    post_r2 = 1 - np.sum((post_covid_actual - post_covid_pred)**2) / np.sum((post_covid_actual - post_covid_actual.mean())**2)
    print(f"\nPre-COVID (Waves 1-4):")
    print(f"  Samples: {len(pre_covid_actual)}")
    print(f"  Mean Actual (log): {pre_covid_actual.mean():.3f}")
    print(f"  Mean Predicted (log): {pre_covid_pred.mean():.3f}")
    print(f"  R²: {pre_r2:.4f}")
    print(f"\nPost-COVID (Wave 5):")
    print(f"  Samples: {len(post_covid_actual)}")
    print(f"  Mean Actual (log): {post_covid_actual.mean():.3f}")
    print(f"  Mean Predicted (log): {post_covid_pred.mean():.3f}")
    print(f"  R²: {post_r2:.4f}")
    pre_bias = (pre_covid_pred - pre_covid_actual).mean()
    post_bias = (post_covid_pred - post_covid_actual).mean()
    print(f"\nPrediction Bias:")
    print(f"  Pre-COVID: {pre_bias:+.4f} ({'Over-predicting' if pre_bias > 0 else 'Under-predicting'})")
    print(f"  Post-COVID: {post_bias:+.4f} ({'Over-predicting' if post_bias > 0 else 'Under-predicting'})")
    if abs(post_bias) > abs(pre_bias) * 1.5:
        print(f"\n⚠️ Model shows {abs(post_bias/pre_bias):.1f}x larger bias in post-COVID period.")
        print(f"   This suggests COVID-19 changed consumption patterns beyond what features capture.")
    else:
        print(f"\n✓ Model performance is consistent across pre and post-COVID periods.")
else:
    print("Wave/COVID information not available in features.")

## 11. Prediction Error by Wealth Level

In [ ]:
print("ERROR ANALYSIS BY WEALTH QUINTILE")
print("=" * 60)
quintile_labels = ['Q1 (Poorest)', 'Q2', 'Q3', 'Q4', 'Q5 (Richest)']
quintiles = pd.qcut(y_test, q=5, labels=quintile_labels)
error_df = pd.DataFrame({'Quintile': quintiles, 'Actual': y_test, 'Predicted': y_pred_log, 'AbsError': np.abs(y_test - y_pred_log), 'PctError': np.abs(y_test - y_pred_log) / (np.abs(y_test) + 0.01) * 100})
quintile_stats = error_df.groupby('Quintile').agg(Count=('Actual', 'count'), Mean_Actual_Log=('Actual', 'mean'), Mean_Predicted_Log=('Predicted', 'mean'), MAE=('AbsError', 'mean'), Median_Pct_Error=('PctError', 'median')).round(4)
display(quintile_stats)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
quintile_stats[['Mean_Actual_Log', 'Mean_Predicted_Log']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Mean Actual vs Predicted by Wealth Quintile')
axes[0].set_ylabel('Log Consumption')
axes[0].set_xlabel('')
axes[0].legend(['Actual', 'Predicted'])
axes[0].tick_params(axis='x', rotation=45)
quintile_stats['Median_Pct_Error'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Median Absolute % Error by Wealth Quintile')
axes[1].set_ylabel('Median |% Error|')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)
plt.suptitle('Model Performance Across Wealth Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Save Best Model for Dashboard

In [ ]:
MODELS_PATH = '../models/'
os.makedirs(MODELS_PATH, exist_ok=True)
joblib.dump(predictor.best_model, f'{MODELS_PATH}best_model.pkl')
print(f"✓ Saved: {MODELS_PATH}best_model.pkl")
feature_names = X_train.columns.tolist()
joblib.dump(feature_names, f'{MODELS_PATH}feature_names.pkl')
print(f"✓ Saved: {MODELS_PATH}feature_names.pkl")
scaler_src = f'{PROCESSED_PATH}scaler.pkl'
if os.path.exists(scaler_src):
    import shutil
    shutil.copy(scaler_src, f'{MODELS_PATH}scaler.pkl')
    print(f"✓ Copied: {MODELS_PATH}scaler.pkl")
print(f"\nModel artifacts ready for dashboard (app/app.py)")

## 13. Generate Summary Report

In [1]:
best_r2 = results[results['Model']==best_model_name]['R2'].values[0]
best_rmse = results[results['Model']==best_model_name]['RMSE'].values[0]
best_cv = results[results['Model']==best_model_name]['CV_R2'].values[0]
y_pred_final = predictor.best_model.predict(X_test)
r2_final = 1 - np.sum((y_test - y_pred_final)**2) / np.sum((y_test - y_test.mean())**2)
report = f"""
╔══════════════════════════════════════════════════════════════════════╗
║                    MODELING SUMMARY REPORT                           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  BEST MODEL: {best_model_name:<55s} ║
║                                                                      ║
║  PERFORMANCE METRICS:                                                ║
║  ├─ Test R²:          {best_r2:.4f}                                  ║
║  ├─ Test RMSE:        {best_rmse:.4f}                                ║
║  ├─ CV R² (mean):     {best_cv:.4f}                                  ║
║  └─ Final R²:         {r2_final:.4f}                                 ║
║                                                                      ║
║  DATA:                                                               ║
║  ├─ Training samples: {X_train.shape[0]:,}                           ║
║  ├─ Test samples:     {X_test.shape[0]:,}                            ║
║  └─ Features:         {X_train.shape[1]}                             ╩

"""
for i, row in results.iterrows():
    report += f"║  {i+1}. {row['Model']:<20s} R²={row['R2']:.4f}  RMSE={row['RMSE']:.4f}              ║\n"
report += """║                                                                      ║
║  SAVED ARTIFACTS:                                                    ║
║  ├─ models/best_model.pkl                                            ║
║  ├─ models/feature_names.pkl                                         ║
║  └─ models/scaler.pkl                                                ║
║                                                                      ║
║  NEXT STEPS:                                                         ║
║  ├─ 04_evaluation.ipynb → Detailed evaluation & robustness checks    ║
║  ├─ 05_dashboard_test.ipynb → Test prediction API                   ║
║  └─ app/app.py → Launch Streamlit dashboard                         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝"""
print(report)
os.makedirs('../reports/', exist_ok=True)
with open('../reports/modeling_report.txt', 'w') as f:
    f.write(report)
print("✓ Report saved to ../reports/modeling_report.txt")

NameError: name 'results' is not defined